In [0]:
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
table_bronze = dbutils.widgets.get("table_bronze")
table_silver = dbutils.widgets.get("table_silver")
primary_keys = dbutils.widgets.get("primary_key")

In [0]:
from pyspark.sql import functions as sf 
from pyspark.sql.dataframe import DataFrame

df_bronze = spark.read.table(f"{catalog}.{schema_bronze}.{table_bronze}")

In [0]:
df_bronze = df_bronze.select("product_id", "product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm")


data_type_mapping = {
    "product_id": "string",
    "product_category_name": "string",
    "product_name_lenght": "integer",
    "product_description_lenght": "integer",
    "product_photos_qty": "integer",
    "product_weight_g": "double",
    "product_length_cm": "double",
    "product_height_cm": "double",
    "product_width_cm": "double"
}

def cast_columns(df: DataFrame, mapping: dict) -> DataFrame:
    existing_columns = df.columns

    for column, data_type in mapping.items():
        if column not in existing_columns:
            raise ValueError(f"Column '{column}' not found in DataFrame")
            

        df = df.withColumn(column, sf.col(column).cast(data_type))
        print(f"Column '{column}' casted to {data_type}")
    
    return df

df_bronze_normalized = cast_columns(df_bronze, data_type_mapping)

In [0]:


def standardardizing_column_name(df: DataFrame) -> DataFrame:
    new_columns = [column.strip().replace(" ", "_").lower() for column in df.columns]
    return df.toDF(*new_columns)

def standardarizing_raw(df: DataFrame) -> DataFrame:
    string_columns = [col_name for col_name, dtype in df.dtypes if dtype == 'string']

    for col in string_columns:
        df = df.withColumn(col, sf.trim(sf.lower(sf.col(col))))
        
    return df
    

df_bronze_column_normalized = standardardizing_column_name(df_bronze_normalized)
df_bronze_standardized = standardarizing_raw(df_bronze_column_normalized)


In [0]:
df_silver = df_bronze_standardized.withColumn("silver_update_date", sf.current_timestamp()).fillna("desconhecido",subset=["product_category_name"])


In [0]:
%run ../utils/utils_merge_into_tables

upsert_data(df_silver, table_silver,primary_keys) # noqa: F821